# 03 — Initial Model Development and Tuning

## Purpose

This notebook develops and compares initial machine-learning models for
predicting concrete compressive strength.

The modeling process will:

1. Load the engineered dataset from Notebook 02.
2. Recreate the same training and test split.
3. Establish a baseline using a dummy regressor.
4. Train initial linear and nonlinear regression models.
5. Compare the models using five-fold cross-validation.
6. Tune the Decision Tree using Grid Search.
7. Select promising models for advanced testing in Notebook 04.

Model selection and tuning will use only the training data. The test set will
remain untouched until final model evaluation.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    cross_validate,
    GridSearchCV
)

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

RANDOM_STATE = 4

In [2]:
df = pd.read_csv(
    "Processed_Data/concrete_cleaned.csv"
)

df.head()

,cement,blast_furnace_slag,fly_ash,water,superplasticizer,coarse_aggregate,fine_aggregate,age,concrete_compressive_strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30


In [3]:
def add_concrete_features(df):
    """
    Add domain-informed features describing concrete mixture composition.

    Parameters
    ----------
    df : pandas.DataFrame
        Cleaned concrete dataset.

    Returns
    -------
    pandas.DataFrame
        Copy of the dataset containing engineered features.
    """
    engineered_df = df.copy()

    engineered_df["binder_total"] = (
        engineered_df["cement"]
        + engineered_df["blast_furnace_slag"]
        + engineered_df["fly_ash"]
    )

    engineered_df["aggregate_total"] = (
        engineered_df["coarse_aggregate"]
        + engineered_df["fine_aggregate"]
    )

    engineered_df["water_binder_ratio"] = (
        engineered_df["water"]
        / engineered_df["binder_total"]
    )

    engineered_df["aggregate_binder_ratio"] = (
        engineered_df["aggregate_total"]
        / engineered_df["binder_total"]
    )

    return engineered_df

In [5]:
df_engineered = add_concrete_features(df)

df_engineered.head()

,cement,blast_furnace_slag,fly_ash,water,superplasticizer,coarse_aggregate,fine_aggregate,age,concrete_compressive_strength,binder_total,aggregate_total,water_binder_ratio,aggregate_binder_ratio
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99,540.0,1716.0,0.30000,3.177778
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89,540.0,1731.0,0.30000,3.205556
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27,475.0,1526.0,0.48000,3.212632
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05,475.0,1526.0,0.48000,3.212632
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30,331.0,1803.9,0.58006,5.449849


In [6]:
engineered_columns = [
    "binder_total",
    "aggregate_total",
    "water_binder_ratio",
    "aggregate_binder_ratio"
]

In [7]:
target_column = "concrete_compressive_strength"

X = df_engineered.drop(columns=target_column)
y = df_engineered[target_column]

In [8]:
print("Predictor shape:", X.shape)
print("Target shape:", y.shape)

display(X.head())
display(y.head())

Predictor shape: (1005, 12)
Target shape: (1005,)


,cement,blast_furnace_slag,fly_ash,water,superplasticizer,coarse_aggregate,fine_aggregate,age,binder_total,aggregate_total,water_binder_ratio,aggregate_binder_ratio
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,540.0,1716.0,0.30000,3.177778
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,540.0,1731.0,0.30000,3.205556
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,475.0,1526.0,0.48000,3.212632
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,475.0,1526.0,0.48000,3.212632
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,331.0,1803.9,0.58006,5.449849


0    79.99
1    61.89
2    40.27
3    41.05
4    44.30
Name: concrete_compressive_strength, dtype: float64

In [10]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

In [11]:
split_comparison = pd.DataFrame({
    "training_set": y_train.describe(),
    "test_set": y_test.describe()
})

split_comparison

,training_set,test_set
count,804.000000,201.00000
mean,35.066978,35.98398
std,16.022924,17.31521
min,3.320000,2.33000
25%,23.222500,23.80000
50%,33.740000,34.57000
75%,44.392500,46.90000
max,79.990000,82.60000


## Modeling Data Preparation

The engineered dataset from Notebook 02 was loaded and divided into predictor
variables and the compressive-strength target.

The same 80/20 train-test split and random state were used to ensure that the
split is reproducible. Model comparison and tuning will use only the training
data, while the test set will remain untouched until final evaluation.

In [12]:
dummy_model = DummyRegressor(strategy="mean")

In [13]:
dummy_cv_results = cross_validate(
    dummy_model,
    X_train,
    y_train,
    cv=5,
    scoring={
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2"
    }
)

In [14]:
dummy_mean_mae = -dummy_cv_results["test_mae"].mean()
dummy_std_mae = dummy_cv_results["test_mae"].std()

dummy_mean_rmse = -dummy_cv_results["test_rmse"].mean()
dummy_mean_r2 = dummy_cv_results["test_r2"].mean()

print(f"Mean CV MAE: {dummy_mean_mae:.2f} MPa")
print(f"CV MAE standard deviation: {dummy_std_mae:.2f}")
print(f"Mean CV RMSE: {dummy_mean_rmse:.2f} MPa")
print(f"Mean CV R²: {dummy_mean_r2:.3f}")

Mean CV MAE: 12.97 MPa
CV MAE standard deviation: 0.50
Mean CV RMSE: 16.03 MPa
Mean CV R²: -0.011


In [15]:
model_results = pd.DataFrame({
    "model": ["Dummy Regressor"],
    "mean_cv_mae": [dummy_mean_mae],
    "cv_mae_std": [dummy_std_mae],
    "mean_cv_rmse": [dummy_mean_rmse],
    "mean_cv_r2": [dummy_mean_r2]
})

model_results

,model,mean_cv_mae,cv_mae_std,mean_cv_rmse,mean_cv_r2
0,Dummy Regressor,12.969578,0.503565,16.031369,-0.010961


## Baseline Model

A dummy regressor was evaluated using five-fold cross-validation on the
training data. The model predicts the mean training-set compressive strength
for every observation and does not use the predictor variables.

Its performance provides a minimum benchmark. A useful machine-learning model
should achieve a lower MAE and RMSE and a higher R² score than this baseline.

In [16]:
linear_model = LinearRegression()

In [17]:
linear_cv_results = cross_validate(
    linear_model,
    X_train,
    y_train,
    cv=5,
    scoring={
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2"
    }
)

In [18]:
linear_mean_mae = -linear_cv_results["test_mae"].mean()
linear_std_mae = linear_cv_results["test_mae"].std()

linear_mean_rmse = -linear_cv_results["test_rmse"].mean()
linear_mean_r2 = linear_cv_results["test_r2"].mean()

print(f"Mean CV MAE: {linear_mean_mae:.2f} MPa")
print(f"CV MAE standard deviation: {linear_std_mae:.2f}")
print(f"Mean CV RMSE: {linear_mean_rmse:.2f} MPa")
print(f"Mean CV R²: {linear_mean_r2:.3f}")

Mean CV MAE: 8.03 MPa
CV MAE standard deviation: 0.53
Mean CV RMSE: 10.11 MPa
Mean CV R²: 0.597


## Linear Regression

Linear Regression was evaluated using the same five-fold cross-validation
procedure as the dummy baseline.

Unlike the dummy model, Linear Regression learns coefficients that describe
the relationship between the predictor variables and concrete compressive
strength. Its results will show how much predictive improvement can be achieved
using a model that assumes the relationships are linear.

In [19]:
tree_model = DecisionTreeRegressor(
    random_state=RANDOM_STATE
)

tree_cv_results = cross_validate(
    tree_model,
    X_train,
    y_train,
    cv=5,
    scoring={
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2"
    }
)

In [24]:
print(linear_mean_mae)
print(linear_std_mae)
print(linear_mean_rmse)
print(linear_mean_r2)

8.032737029912667
0.5273561871065741
10.111537284477478
0.5967156209077012


In [25]:
linear_results = pd.DataFrame({
    "model": ["Linear Regression"],
    "mean_cv_mae": [linear_mean_mae],
    "cv_mae_std": [linear_std_mae],
    "mean_cv_rmse": [linear_mean_rmse],
    "mean_cv_r2": [linear_mean_r2]
})

In [26]:
model_results = pd.concat(
    [model_results, linear_results],
    ignore_index=True
)

model_results

,model,mean_cv_mae,cv_mae_std,mean_cv_rmse,mean_cv_r2
0,Dummy Regressor,12.969578,0.503565,16.031369,-0.010961
1,Linear Regression,8.032737,0.527356,10.111537,0.596716


In [27]:
model_results.sort_values(
    by="mean_cv_mae"
)

,model,mean_cv_mae,cv_mae_std,mean_cv_rmse,mean_cv_r2
1,Linear Regression,8.032737,0.527356,10.111537,0.596716
0,Dummy Regressor,12.969578,0.503565,16.031369,-0.010961


In [21]:
tree_parameter_grid = {
    "max_depth": [3, 5, 8, 12, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

tree_grid_search = GridSearchCV(
    estimator=DecisionTreeRegressor(
        random_state=RANDOM_STATE
    ),
    param_grid=tree_parameter_grid,
    scoring="neg_mean_absolute_error",
    cv=5,
    n_jobs=-1
)

tree_grid_search.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=DecisionTreeRegressor(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [3, 5, 8, 12, None],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10]},
             scoring='neg_mean_absolute_error')

In [22]:
print("Best parameters:")
print(tree_grid_search.best_params_)

print(
    "Best cross-validated MAE:",
    -tree_grid_search.best_score_
)

Best parameters:
{'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2}
Best cross-validated MAE: 4.15341793478261


In [28]:
tree_parameter_grid = {
    "max_depth": [3, 5, 8, 12, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

tree_grid_search = GridSearchCV(
    estimator=DecisionTreeRegressor(
        random_state=RANDOM_STATE
    ),
    param_grid=tree_parameter_grid,
    scoring="neg_mean_absolute_error",
    cv=5,
    n_jobs=-1
)

tree_grid_search.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=DecisionTreeRegressor(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [3, 5, 8, 12, None],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10]},
             scoring='neg_mean_absolute_error')

In [29]:
tree_model = DecisionTreeRegressor(
    random_state=RANDOM_STATE
)

tree_cv_results = cross_validate(
    tree_model,
    X_train,
    y_train,
    cv=5,
    scoring={
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2"
    }
)

In [30]:
tree_mean_mae = -tree_cv_results["test_mae"].mean()
tree_std_mae = tree_cv_results["test_mae"].std()
tree_mean_rmse = -tree_cv_results["test_rmse"].mean()
tree_mean_r2 = tree_cv_results["test_r2"].mean()

print(f"Mean CV MAE: {tree_mean_mae:.2f} MPa")
print(f"CV MAE standard deviation: {tree_std_mae:.2f}")
print(f"Mean CV RMSE: {tree_mean_rmse:.2f} MPa")
print(f"Mean CV R²: {tree_mean_r2:.3f}")

Mean CV MAE: 4.15 MPa
CV MAE standard deviation: 0.10
Mean CV RMSE: 6.10 MPa
Mean CV R²: 0.852


In [31]:
tree_parameter_grid = {
    "max_depth": [3, 5, 8, 12, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

In [32]:
tree_grid_search = GridSearchCV(
    estimator=DecisionTreeRegressor(
        random_state=RANDOM_STATE
    ),
    param_grid=tree_parameter_grid,
    scoring="neg_mean_absolute_error",
    cv=5,
    n_jobs=-1
)

tree_grid_search.fit(
    X_train,
    y_train
)

GridSearchCV(cv=5, estimator=DecisionTreeRegressor(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [3, 5, 8, 12, None],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10]},
             scoring='neg_mean_absolute_error')

In [33]:
tuned_tree_model = tree_grid_search.best_estimator_

tuned_tree_model

DecisionTreeRegressor(random_state=42)

In [35]:
tuned_tree_cv_results = cross_validate(
    tuned_tree_model,
    X_train,
    y_train,
    cv=5,
    scoring={
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2"
    }
)

In [36]:
tuned_tree_mean_mae = (
    -tuned_tree_cv_results["test_mae"].mean()
)

tuned_tree_std_mae = (
    tuned_tree_cv_results["test_mae"].std()
)

tuned_tree_mean_rmse = (
    -tuned_tree_cv_results["test_rmse"].mean()
)

tuned_tree_mean_r2 = (
    tuned_tree_cv_results["test_r2"].mean()
)

print(f"Mean CV MAE: {tuned_tree_mean_mae:.2f} MPa")
print(f"CV MAE standard deviation: {tuned_tree_std_mae:.2f}")
print(f"Mean CV RMSE: {tuned_tree_mean_rmse:.2f} MPa")
print(f"Mean CV R²: {tuned_tree_mean_r2:.3f}")

Mean CV MAE: 4.15 MPa
CV MAE standard deviation: 0.10
Mean CV RMSE: 6.10 MPa
Mean CV R²: 0.852


In [37]:
model_results = pd.DataFrame({
    "model": [
        "Dummy Regressor",
        "Linear Regression",
        "Decision Tree",
        "Tuned Decision Tree"
    ],
    "mean_cv_mae": [
        dummy_mean_mae,
        linear_mean_mae,
        tree_mean_mae,
        tuned_tree_mean_mae
    ],
    "cv_mae_std": [
        dummy_std_mae,
        linear_std_mae,
        tree_std_mae,
        tuned_tree_std_mae
    ],
    "mean_cv_rmse": [
        dummy_mean_rmse,
        linear_mean_rmse,
        tree_mean_rmse,
        tuned_tree_mean_rmse
    ],
    "mean_cv_r2": [
        dummy_mean_r2,
        linear_mean_r2,
        tree_mean_r2,
        tuned_tree_mean_r2
    ]
})

model_results.sort_values(
    by="mean_cv_mae"
)

,model,mean_cv_mae,cv_mae_std,mean_cv_rmse,mean_cv_r2
3,Tuned Decision Tree,4.153418,0.103131,6.104794,0.852322
2,Decision Tree,4.153418,0.103131,6.104794,0.852322
1,Linear Regression,8.032737,0.527356,10.111537,0.596716
0,Dummy Regressor,12.969578,0.503565,16.031369,-0.010961
